In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 83.2 MB/s eta 0:00:00


##PART 1

In [ ]:
from gensim.models import KeyedVectors


wv = KeyedVectors.load('embs_train.kv')
print(wv)
print()


print(wv['big'])

print()

print(wv.most_similar('dog', topn=10))


print(wv.most_similar('man', topn=10))

print()
print(wv.most_similar('wonderful', topn=10))
print()

print(wv.most_similar('awful', topn=10))
print()

print(wv.most_similar('boring', topn=10))
print()

print(wv.most_similar('excellent', topn=10))
print()

print(wv.most_similar('fight', topn=10))
print()

print(wv.most_similar(positive=['bigger', 'good'], negative=['big'], topn=5))
print()

print(wv.most_similar(positive=['woman', 'king'], negative=['man'], topn=5))
print()


print(wv.most_similar(positive=['sister', 'man'], negative=['woman'], topn=10))
print()


print(wv.most_similar(positive=['harder', 'fast'], negative=['hard'], topn=10))
print()


print(wv.most_similar(positive=['good', 'bad'], negative=['ugly'], topn=10))
print()

print(wv.most_similar(positive=['decent', 'intelligent'], negative=['dumb'], topn=10))
print()

print(wv.most_similar(positive=['suspect', 'victim'], negative=['thief'], topn=10))
print()

KeyedVectors<vector_size=300, 14414 keys>

[ 0.11132812  0.10595703 -0.07373047  0.18847656  0.07666016 -0.3828125
 -0.0625     -0.07470703  0.05957031  0.22167969  0.20507812 -0.09228516
  0.05395508  0.01379395 -0.16992188  0.05493164  0.09619141  0.06103516
 -0.14160156  0.03173828 -0.08642578  0.12011719  0.06445312  0.22070312
  0.06835938  0.04956055 -0.22460938 -0.06298828  0.09179688 -0.00531006
 -0.11425781  0.20605469  0.31054688 -0.0625     -0.02026367 -0.13476562
 -0.02697754  0.2734375   0.27929688  0.21386719  0.25195312 -0.13964844
  0.19824219 -0.07421875  0.09228516  0.125       0.0612793  -0.02990723
  0.0072937  -0.05615234 -0.08447266  0.1796875  -0.17578125 -0.11328125
 -0.17578125 -0.1171875   0.09082031 -0.07177734  0.30273438 -0.2734375
 -0.07128906  0.33007812 -0.13574219 -0.0390625   0.01397705 -0.02526855
  0.05981445  0.14550781 -0.11035156  0.12988281  0.12695312 -0.04980469
  0.16992188  0.18261719 -0.23144531  0.07910156 -0.06738281  0.34960938
 -0.073242

##PART 2

In [ ]:
(wv['the'] + wv['man'] + wv['bit'] + wv['the'] + wv['dog'])/5


array([ 1.51464850e-01,  9.12353545e-02, -4.32617180e-02,  8.23242217e-02,
       -2.34497078e-02, -2.36328132e-02,  3.10546868e-02, -8.69018584e-02,
        1.93847660e-02,  6.26449585e-02,  2.28515621e-02, -1.92578122e-01,
       -2.71484368e-02, -1.12695314e-01, -9.51660126e-02,  1.31347654e-02,
       -5.90820312e-02,  1.01025388e-01, -1.40136722e-02, -6.15722649e-02,
       -9.96093731e-03,  4.84039299e-02,  1.57714840e-02, -9.12261978e-02,
        5.64453118e-02,  3.21777351e-02, -9.22851562e-02,  1.58740237e-01,
        1.83294684e-01,  6.33789077e-02,  1.25000002e-02,  5.63964844e-02,
       -1.88964847e-02, -4.11743149e-02, -5.76171884e-03,  5.84823601e-02,
        4.01855484e-02, -8.20312463e-03,  1.05859376e-01,  1.91406250e-01,
        3.95996086e-02, -1.55371100e-01,  1.12219237e-01,  1.20971678e-02,
        3.56201157e-02, -9.12475586e-03, -1.67968757e-02,  4.03808579e-02,
        4.86572273e-02,  6.51367158e-02, -9.76562500e-03,  7.69287124e-02,
        5.32958992e-02, -

In [ ]:
# 2 Better Perceptron using Embeddings


import numpy as np
import pandas as pd
from gensim.models import KeyedVectors
from collections import defaultdict
from sklearn.feature_extraction import DictVectorizer
from sklearn.neighbors import KNeighborsClassifier
import time


wv = KeyedVectors.load('embs_train.kv')

def read_from(csv_path):

    df = pd.read_csv(csv_path)

    for i in range(len(df)):
        _id, words, label = df.iloc[i]

        y = 1 if label == "+" else -1

        yield y, words.split(), _id

def make_sentence_embedding(words):

    vectors = []

    for word in words:

        if word in wv:

            vectors.append(wv[word])


    if len(vectors) == 0:

        return np.zeros(wv.vector_size)

    return np.mean(vectors, axis=0)

def cosine_similarity(v1, v2):

    dot = np.dot(v1, v2)

    norm1 = np.linalg.norm(v1)

    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:

        return 0

    return dot / (norm1 * norm2)

def euclidean_distance(v1, v2):

    return np.linalg.norm(v1 - v2)


def make_onehot_vector(words):

    vec = defaultdict(float)
    for word in words:

        vec[word] += 1

    return vec

def make_sentence_embedding_pruned(words, word_freq, threshold=1):

    vectors = [wv[w] for w in words if word_freq.get(w, 0) > threshold and w in wv]

    return np.mean(vectors, axis=0) if vectors else np.zeros(wv.vector_size)

def onehot_distance(v1, v2):

    all_keys = set(v1.keys()) | set(v2.keys())

    return np.sqrt(sum((v1.get(k, 0) - v2.get(k, 0))**2 for k in all_keys))

def test_error(dev_csv, w, use_pruning=False, word_freq=None):

    errors = 0

    total = 0

    for y, words, _id in read_from(dev_csv):
        x = make_sentence_embedding_pruned(words, word_freq) if use_pruning else make_sentence_embedding(words)

        if y * np.dot(w, x) <= 0:

            errors += 1

        total += 1

    return errors / total

def count_word_frequencies(train_csv):

    freq = {}

    for y, words, _id in read_from(train_csv):

        for w in words:
            freq[w] = freq.get(w, 0) + 1

    return freq



In [ ]:
# 2.1 Sentence Embedding and k-NN

train_data_emb = [(y, make_sentence_embedding(words), words, _id) for y, words, _id in read_from('train.csv')]

dev_data_emb = [(y, make_sentence_embedding(words), words, _id) for y, words, _id in read_from('dev.csv')]

train_data_onehot = [(y, make_onehot_vector(words), words, _id) for y, words, _id in read_from('train.csv')]

dev_data_onehot = [(y, make_onehot_vector(words), words, _id) for y, words, _id in read_from('dev.csv')]


print("Q1: \n")

first_pos = next((i, d) for i, d in enumerate(train_data_emb) if d[0] == 1)

idx, (y, emb, words, _id) = first_pos

print(f"First positive (id={_id}):")

print(" ".join(words)[:100])

best_sim, best_idx = max(((cosine_similarity(emb, d[1]), i) for i, d in enumerate(train_data_emb) if i != idx), key=lambda x: x[0])

print(f"\nClosest (id={train_data_emb[best_idx][3]}, label={'+' if train_data_emb[best_idx][0]==1 else '-'}):")

print(" ".join(train_data_emb[best_idx][2])[:100])

print(f"Similarity: {best_sim:.4f}\n")


print("Q2:\n")

neg_examples = [(i, d) for i, d in enumerate(train_data_emb) if d[0] == -1]

idx, (y, emb, words, _id) = neg_examples[1]

print(f"Second negative (id={_id}):")

print(" ".join(words)[:100])

best_sim, best_idx = max(((cosine_similarity(emb, d[1]), i) for i, d in enumerate(train_data_emb) if i != idx), key=lambda x: x[0])

print(f"\nClosest (id={train_data_emb[best_idx][3]}, label={'+' if train_data_emb[best_idx][0]==1 else '-'}):")

print(" ".join(train_data_emb[best_idx][2])[:100])

print(f"Similarity: {best_sim:.4f}\n")


print("Q3:\n")

X_train = np.array([emb for y, emb, _, _ in train_data_emb])

y_train = np.array([y for y, emb, _, _ in train_data_emb])

X_dev = np.array([emb for y, emb, _, _ in dev_data_emb])

y_dev = np.array([y for y, emb, _, _ in dev_data_emb])

k_values = list(range(1, 100, 2))

embedding_errors = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)

    knn.fit(X_train, y_train)

    score = knn.score(X_dev, y_dev)

    err = 1 - score

    embedding_errors.append(err)

    if k <= 11 or k % 10 == 1:

        print(f"k={k:2d}: {err*100:.2f}%")


best_k_emb = k_values[np.argmin(embedding_errors)]

best_err_emb = min(embedding_errors)

print(f"\n Best k-NN (embeddings): k={best_k_emb}, error={best_err_emb*100:.2f}%\n")

print("Q4:\n")

vec = DictVectorizer()

X_train_onehot = vec.fit_transform([v for y, v, _, _ in train_data_onehot])

X_dev_onehot = vec.transform([v for y, v, _, _ in dev_data_onehot])

onehot_errors = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)

    knn.fit(X_train_onehot, y_train)

    score = knn.score(X_dev_onehot, y_dev)

    err = 1 - score

    onehot_errors.append(err)

    if k <= 11 or k % 10 == 1:
        print(f"k={k:2d}: {err*100:.2f}%")

best_k_onehot = k_values[np.argmin(onehot_errors)]

best_err_onehot = min(onehot_errors)

print(f"\n Best k-NN (one-hot): k={best_k_onehot}, error={best_err_onehot*100:.2f}%\n")


Q1: 

First positive (id=0):
it 's a tour de force , written and directed so quietly that it 's implosion rather than explosion y

Closest (id=2061, label=-):
a semi autobiographical film that 's so sloppily written and cast that you can not believe anyone mo
Similarity: 0.7771

Q2:

Second negative (id=3):
exactly what you 'd expect from a guy named kaos

Closest (id=1191, label=-):
it 's exactly what you 'd expect
Similarity: 0.9121

Q3:

k= 1: 37.00%
k= 3: 34.80%
k= 5: 34.50%
k= 7: 33.80%
k= 9: 31.90%
k=11: 30.70%
k=21: 30.90%
k=31: 29.60%
k=41: 30.40%
k=51: 28.90%
k=61: 28.50%
k=71: 28.90%
k=81: 29.00%
k=91: 28.80%

 Best k-NN (embeddings): k=73, error=27.80%

Q4:

k= 1: 43.30%
k= 3: 40.50%
k= 5: 41.90%
k= 7: 41.10%
k= 9: 40.20%
k=11: 41.90%
k=21: 43.50%
k=31: 41.70%
k=41: 42.80%
k=51: 44.70%
k=61: 43.40%
k=71: 44.20%
k=81: 43.20%
k=91: 44.20%

 Best k-NN (one-hot): k=9, error=40.20%



In [ ]:
print("Q5: \n")

test_data_emb = [(make_sentence_embedding(words), _id) for y, words, _id in read_from('test.csv')]

X_test = np.array([emb for emb, _ in test_data_emb])
test_ids = [_id for _, _id in test_data_emb]

best_knn = KNeighborsClassifier(n_neighbors=best_k_emb)

best_knn.fit(X_train, y_train)


y_pred = best_knn.predict(X_test)


predictions = ['+' if pred == 1 else '-' for pred in y_pred]


submission = pd.DataFrame({ 'id': test_ids, 'target': predictions })


submission.to_csv('knn_submission.csv', index=False)

print(f"Saved 'knn_submission.csv'")

Q5: 

Saved 'knn_submission.csv'


In [ ]:
# 2.2 Reimplement Perceptron
print("\nQ1:")


def make_sentence_embedding_with_bias(words):
    vectors = [wv[w] for w in words if w in wv]

    emb = np.mean(vectors, axis=0) if vectors else np.zeros(wv.vector_size)

    return np.append(emb, 1.0)

def test_error(dev_csv, w):

    errors = 0

    total = 0

    for y, words, _id in read_from(dev_csv):
        x = make_sentence_embedding_with_bias(words)

        if y * np.dot(w, x) <= 0:

            errors += 1

        total += 1

    return errors / total

def train_perceptron(train_csv, dev_csv, epochs=10):

    w = np.zeros(wv.vector_size + 1)

    best_err = 1.0

    t = time.time()

    for epoch in range(1, epochs + 1):
        updates = 0

        total = 0

        for y, words, _id in read_from(train_csv):
            total += 1

            x = make_sentence_embedding_with_bias(words)


            if y * np.dot(w, x) <= 0:

                updates += 1

                w += y * x


        dev_err = test_error(dev_csv, w)

        best_err = min(best_err, dev_err)

        print(f"epoch {epoch}, update {updates/total*100:.1f}%, dev {dev_err*100:.1f}%")

    print(f"\nbest dev err {best_err*100:.1f}%, time: {time.time()-t:.1f} secs")

    print(f"HW2 basic perceptron (one-hot): ~28.9%")

    print(f"HW4 basic perceptron (embeddings): {best_err*100:.1f}%\n")

    return w, best_err

w_basic, best_err_basic = train_perceptron('train.csv', 'dev.csv', epochs=10)

print("\n Q2: \n")
def train_averaged_perceptron(train_csv, dev_csv, epochs=10):

    w = np.zeros(wv.vector_size + 1)

    u = np.zeros(wv.vector_size + 1)

    c = 0

    best_err = 1.0

    best_model = None

    dev_history = []

    t = time.time()

    for epoch in range(1, epochs + 1):

        updates = 0

        total = 0

        for y, words, _id in read_from(train_csv):

            total += 1

            c += 1

            x = make_sentence_embedding_with_bias(words)



            if y * np.dot(w, x) <= 0:

                updates += 1

                w += y * x
                u += y * (c * x)



        w_avg = w - (1.0 / c) * u


        dev_err = test_error(dev_csv, w_avg)

        dev_history.append(dev_err)

        if dev_err <= best_err:

            best_err = dev_err
            best_model = w_avg.copy()


        print(f"epoch {epoch}, update {updates/total*100:.1f}%, dev {dev_err*100:.1f}%")

    print(f"\nbest dev err {best_err*100:.1f}%, time: {time.time()-t:.1f} secs")

    print(f"\nHW2 averaged perceptron (one-hot): ~26.3%")

    print(f"HW4 averaged perceptron (embeddings): {best_err*100:.1f}%\n")

    return best_model, best_err


w_avg, best_err_avg = train_averaged_perceptron('train.csv', 'dev.csv', epochs=10)


Q1:
epoch 1, update 32.3%, dev 32.7%
epoch 2, update 30.2%, dev 36.2%
epoch 3, update 29.2%, dev 39.4%
epoch 4, update 28.8%, dev 41.6%
epoch 5, update 29.3%, dev 33.7%
epoch 6, update 28.7%, dev 31.3%
epoch 7, update 29.3%, dev 36.2%
epoch 8, update 28.7%, dev 39.1%
epoch 9, update 28.9%, dev 41.8%
epoch 10, update 28.8%, dev 39.0%

best dev err 31.3%, time: 12.5 secs
HW2 basic perceptron (one-hot): ~28.9%
HW4 basic perceptron (embeddings): 31.3%


 Q2: 

epoch 1, update 32.3%, dev 25.4%
epoch 2, update 30.2%, dev 25.3%
epoch 3, update 29.2%, dev 25.2%
epoch 4, update 28.8%, dev 25.0%
epoch 5, update 29.3%, dev 24.2%
epoch 6, update 28.7%, dev 24.3%
epoch 7, update 29.3%, dev 24.5%
epoch 8, update 28.7%, dev 24.4%
epoch 9, update 28.9%, dev 24.3%
epoch 10, update 28.8%, dev 24.6%

best dev err 24.2%, time: 12.0 secs
dev history (%): [25.4, 25.3, 25.2, 25.0, 24.2, 24.3, 24.5, 24.4, 24.3, 24.6]

HW2 averaged perceptron (one-hot): ~26.3%
HW4 averaged perceptron (embeddings): 24.2%



In [ ]:
print("Q4:\n")


def count_word_frequencies(train_csv):

    word_freq = {}

    for y, words, _id in read_from(train_csv):

        for word in words:

            word_freq[word] = word_freq.get(word, 0) + 1

    return word_freq

word_freq = count_word_frequencies('train.csv')

vocab_size = len(word_freq)

one_count_words = sum(1 for count in word_freq.values() if count == 1)


def make_sentence_embedding_pruned(words, word_freq, threshold=1):

    vectors = [wv[w] for w in words if word_freq.get(w, 0) > threshold and w in wv]

    emb = np.mean(vectors, axis=0) if vectors else np.zeros(wv.vector_size)

    return np.append(emb, 1.0)

def test_error_pruned(dev_csv, w, word_freq):

    errors = 0

    total = 0

    for y, words, _id in read_from(dev_csv):

        x = make_sentence_embedding_pruned(words, word_freq)

        if y * np.dot(w, x) <= 0:

            errors += 1

        total += 1

    return errors / total

def train_averaged_perceptron_pruned(train_csv, dev_csv, word_freq, epochs=10):

    w = np.zeros(wv.vector_size + 1)

    u = np.zeros(wv.vector_size + 1)

    c = 0

    best_err = 1.0

    best_model = None

    dev_history = []

    t = time.time()


    for epoch in range(1, epochs + 1):

        updates = 0

        total = 0

        for y, words, _id in read_from(train_csv):

            total += 1
            c += 1

            x = make_sentence_embedding_pruned(words, word_freq)

            if y * np.dot(w, x) <= 0:
                updates += 1

                w += y * x

                u += y * (c * x)

        w_avg = w - (1.0 / c) * u
        dev_err = test_error_pruned(dev_csv, w_avg, word_freq)

        dev_history.append(dev_err)

        if dev_err <= best_err:

            best_err = dev_err

            best_model = w_avg.copy()

        print(f"epoch {epoch}, update {updates/total*100:.1f}%, dev {dev_err*100:.1f}%")

    print(f"\nbest dev err {best_err*100:.1f}%, time: {time.time()-t:.1f} secs")

    print(f"dev history (%): {[round(e*100, 1) for e in dev_history]}")

    print(f"\nHW2 avg perceptron + pruning (one-hot): ~25.5%")

    print(f"HW4 avg perceptron + pruning (embeddings): {best_err*100:.1f}%\n")

    return best_model, best_err

# Train with pruning
w_avg_pruned, best_err_pruned = train_averaged_perceptron_pruned('train.csv', 'dev.csv', word_freq, epochs=10)

Q4:

epoch 1, update 33.5%, dev 24.4%
epoch 2, update 30.7%, dev 24.8%
epoch 3, update 30.3%, dev 23.5%
epoch 4, update 30.4%, dev 23.5%
epoch 5, update 30.0%, dev 24.1%
epoch 6, update 30.7%, dev 23.9%
epoch 7, update 30.3%, dev 23.7%
epoch 8, update 30.3%, dev 23.8%
epoch 9, update 30.3%, dev 23.9%
epoch 10, update 30.3%, dev 24.0%

best dev err 23.5%, time: 12.2 secs
dev history (%): [24.4, 24.8, 23.5, 23.5, 24.1, 23.9, 23.7, 23.8, 23.9, 24.0]

HW2 avg perceptron + pruning (one-hot): ~25.5%
HW4 avg perceptron + pruning (embeddings): 23.5%



In [ ]:
print("\n Q6: \n")

print("averaged perceptron (no pruning)")

predictions = []

for y, words, _id in read_from('test.csv'):

    x = make_sentence_embedding_with_bias(words)

    pred = 1 if np.dot(w_avg, x) > 0 else -1

    label = '+' if pred == 1 else '-'

    predictions.append([_id, label])



df = pd.DataFrame(predictions, columns=['id', 'target'])

df.to_csv('perceptron_avg_submission.csv', index=False)

print(f"  Dev error: {best_err_avg*100:.1f}%\n")


print("averaged perceptron (with pruning)")

predictions = []

for y, words, _id in read_from('test.csv'):

    x = make_sentence_embedding_pruned(words, word_freq)

    pred = 1 if np.dot(w_avg_pruned, x) > 0 else -1

    label = '+' if pred == 1 else '-'

    predictions.append([_id, label])


df = pd.DataFrame(predictions, columns=['id', 'target'])

df.to_csv('perceptron_avg_pruned_submission.csv', index=False)

print(f"  Dev error: {best_err_pruned*100:.1f}%\n")

print("Upload both to Kaggle and record your public error rates!")



 Q6: 

averaged perceptron (no pruning)
  Dev error: 24.2%

averaged perceptron (with pruning)
  Dev error: 23.5%

Upload both to Kaggle and record your public error rates!


In [ ]:
!pip install tabulate

In [ ]:
# 2.3 Summarize the error rates in table

from tabulate import tabulate

knn_kaggle_public = 26.2

perceptron_kaggle_public = 22.4


summary_data = {
    'Method': ['k-NN', 'perceptron'],
    'one-hot - best dev': [f'{best_err_onehot*100:.1f}%', '26.3%'],
    'embedding - best dev': [f'{best_err_emb*100:.1f}%', f'{best_err_avg*100:.1f}%'],
    'best kaggle public': [f'{knn_kaggle_public:.1f}%', f'{perceptron_kaggle_public:.1f}%']
}

summary_table = pd.DataFrame(summary_data)

summary_table.to_string(index=False)

print(tabulate(summary_table, headers='keys', tablefmt='grid', showindex=False))

print("\n")


+------------+----------------------+------------------------+----------------------+
| Method     | one-hot - best dev   | embedding - best dev   | best kaggle public   |
+============+======================+========================+======================+
| k-NN       | 40.2%                | 27.8%                  | 26.2%                |
+------------+----------------------+------------------------+----------------------+
| perceptron | 26.3%                | 24.2%                  | 22.4%                |
+------------+----------------------+------------------------+----------------------+




## Part 3 Other Learning Algorithms

In [ ]:
from sklearn.svm import SVC

X_train_emb = np.array([make_sentence_embedding_pruned(words, word_freq) for y, words, _id in read_from('train.csv')])

y_train_emb = np.array([y for y, words, _id in read_from('train.csv')])

X_dev_emb = np.array([make_sentence_embedding_pruned(words, word_freq) for y, words, _id in read_from('dev.csv')])

y_dev_emb = np.array([y for y, words, _id in read_from('dev.csv')])

t = time.time()

clf = SVC(kernel='rbf', C=1.0, random_state=42)

clf.fit(X_train_emb, y_train_emb)

train_time = time.time() - t



dev_acc = clf.score(X_dev_emb, y_dev_emb)

dev_err = 1 - dev_acc

print(f"Dev accuracy: {dev_acc*100:.2f}%")

print(f"Dev error: {dev_err*100:.2f}%\n")

print(f"  Averaged Perceptron (pruned): {best_err_pruned*100:.1f}% dev error")

print(f"  Support Vector Machine: {dev_err*100:.1f}% dev error, {train_time:.1f} secs ")




Dev accuracy: 76.60%
Dev error: 23.40%

  Averaged Perceptron (pruned): 23.5% dev error
  Support Vector Machine: 23.4% dev error, 6.0 secs 
